In [ ]:
#1 读取数据
import scanpy as sc
import scvelo as scv
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import os

os.makedirs("Results", exist_ok=True)

adata_scvelo = sc.read_h5ad("../data/scvelo_output_2.h5ad")
adata_deepvelo = sc.read_h5ad("../data/deepvelo_output_2.h5ad")
adata_transvelo = sc.read_h5ad("../transvelo_output_6.h5ad")

In [ ]:
#2 统一velocity layers
adata_scvelo.layers["velocity_model"] = adata_scvelo.layers["velocity"]
adata_deepvelo.layers["velocity_model"] = adata_deepvelo.layers["velocity"]
adata_transvelo.layers["velocity_model"] = adata_transvelo.layers["transvelo_velocity"]

In [ ]:
#3 for adata in [adata_scvelo, adata_deepvelo, adata_transvelo]:

#scv.tl.velocity_confidence(adata)
for adata in [adata_scvelo, adata_deepvelo, adata_transvelo]:

    V = adata.layers["velocity_model"]

    V = np.nan_to_num(V)

    adata.layers["velocity_model"] = V

    adata.obs["velocity_strength"] = np.linalg.norm(V, axis=1)

In [ ]:
#4 清理 NaN + 计算 velocity magnitude
for adata in [adata_scvelo, adata_deepvelo, adata_transvelo]:

    scv.pp.neighbors(adata)

    scv.tl.velocity_graph(adata)

In [ ]:
#5 Velocity Confidence
for adata in [adata_scvelo, adata_deepvelo, adata_transvelo]:

    scv.tl.velocity_confidence(adata)

In [ ]:
#6 Velocity Pseudotime
for adata in [adata_scvelo, adata_deepvelo, adata_transvelo]:

    scv.tl.velocity_pseudotime(adata)

In [ ]:
#7 Velocity Coherence
def velocity_coherence(adata):

    V = adata.layers["velocity_model"]

    neighbors = adata.obsp["connectivities"]

    coherence = []

    for i in range(V.shape[0]):

        neigh_idx = neighbors[i].indices

        if len(neigh_idx) == 0:
            coherence.append(np.nan)
            continue

        v = V[i]

        v_neighbors = V[neigh_idx]

        cos_sim = np.dot(v_neighbors, v) / (
            np.linalg.norm(v_neighbors, axis=1) *
            np.linalg.norm(v) + 1e-8
        )

        coherence.append(np.nanmean(cos_sim))

    adata.obs["velocity_coherence"] = coherence


for adata in [adata_scvelo, adata_deepvelo, adata_transvelo]:

    velocity_coherence(adata)

In [ ]:
#8 Trajectory Smoothness
def trajectory_smoothness(adata):

    V = adata.layers["velocity_model"]

    neighbors = adata.obsp["connectivities"]

    smoothness = []

    for i in range(V.shape[0]):

        neigh_idx = neighbors[i].indices

        if len(neigh_idx) == 0:

            smoothness.append(np.nan)

            continue

        diff = V[i] - V[neigh_idx]

        smoothness.append(np.mean(np.linalg.norm(diff, axis=1)))

    adata.obs["trajectory_smoothness"] = smoothness


for adata in [adata_scvelo, adata_deepvelo, adata_transvelo]:

    trajectory_smoothness(adata)

In [ ]:
#9 Velocity-Pseudotime Correlation
def velocity_pt_corr(adata):

    v = np.linalg.norm(
        adata.layers["velocity_model"],
        axis=1
    )

    pt = adata.obs["velocity_pseudotime"]

    corr, _ = spearmanr(v, pt)

    return corr

In [ ]:
#10 Velocity Field 图
fig, axes = plt.subplots(1,3, figsize=(18,6))

scv.pl.velocity_embedding_stream(
    adata_scvelo,
    basis="umap",
    color="celltype.anno",
    ax=axes[0],
    title="scVelo",
    show=False
)

scv.pl.velocity_embedding_stream(
    adata_deepvelo,
    basis="umap",
    color="celltype.anno",
    ax=axes[1],
    title="DeepVelo",
    show=False
)

scv.pl.velocity_embedding_stream(
    adata_transvelo,
    basis="umap",
    color="celltype.anno",
    ax=axes[2],
    title="TransVelo",
    show=False
)

plt.tight_layout()

plt.savefig("Results/1_velocity_field.png", dpi=600)

plt.show()

In [ ]:
#计算latent time correlation
import scvelo as scv

for adata in [adata_scvelo, adata_deepvelo, adata_transvelo]:
    scv.tl.recover_dynamics(adata)
    scv.tl.latent_time(adata)

In [ ]:
adata.obs["latent_time"]

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1,3, figsize=(18,6))

scv.pl.umap(
    adata_scvelo,
    color="latent_time",
    ax=axes[0],
    title="scVelo latent time",
    show=False
)

scv.pl.umap(
    adata_deepvelo,
    color="latent_time",
    ax=axes[1],
    title="DeepVelo latent time",
    show=False
)

scv.pl.umap(
    adata_transvelo,
    color="latent_time",
    ax=axes[2],
    title="TransVelo latent time",
    show=False
)

plt.tight_layout()

plt.savefig(
    "Results/latent_time_comparison.png",
    dpi=600
)

plt.show()

In [ ]:
""" #6 论文图1：velocity field
scv.pl.velocity_embedding_stream(
    adata_scvelo,
    basis="umap",
    title="scVelo"
)

scv.pl.velocity_embedding_stream(
    adata_deepvelo,
    basis="umap",
    title="DeepVelo"
)

scv.pl.velocity_embedding_stream(
    adata_transvelo,
    basis="umap",
    title="TransVelo"
) """

import scvelo as scv
import matplotlib.pyplot as plt
import os
# 翻转 scVelo UMAP 的 x 轴
adata_scvelo.obsm["X_umap"][:,0] *= -1
# 创建输出目录
os.makedirs("Results", exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18,6))

# scVelo
scv.pl.velocity_embedding_stream(
    adata_scvelo,
    basis="umap",
    color="celltype.anno",
    ax=axes[0],
    title="scVelo",
    show=False
)

# DeepVelo
scv.pl.velocity_embedding_stream(
    adata_deepvelo,
    basis="umap",
    color="celltype.anno",
    ax=axes[1],
    title="DeepVelo",
    show=False
)

# TransVelo
scv.pl.velocity_embedding_stream(
    adata_transvelo,
    basis="umap",
    color="celltype.anno",
    ax=axes[2], 
    title="TransVelo",
    show=False
)

plt.tight_layout()

plt.savefig(
    "3-1/1-velocity_field_comparison.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
""" #6 论文图1：velocity field
scv.pl.velocity_embedding_stream(
    adata_scvelo,
    basis="umap",
    title="scVelo"
)

scv.pl.velocity_embedding_stream(
    adata_deepvelo,
    basis="umap",
    title="DeepVelo"
)

scv.pl.velocity_embedding_stream(
    adata_transvelo,
    basis="umap",
    title="TransVelo"
) """

import scvelo as scv
import matplotlib.pyplot as plt
import os
# 翻转 scVelo UMAP 的 x 轴
adata_scvelo.obsm["X_umap"][:,0] *= -1
# 创建输出目录
os.makedirs("Results", exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18,6))

# scVelo
scv.pl.velocity_embedding_stream(
    adata_scvelo,
    basis="umap",
    color="condition",
    ax=axes[0],
    title="scVelo",
    title_fontsize=18,
    show=False
)

# DeepVelo
scv.pl.velocity_embedding_stream(
    adata_deepvelo,
    basis="umap",
    color="condition",
    ax=axes[1],
    title="DeepVelo",
    title_fontsize=18,
    show=False
)

# TransVelo
scv.pl.velocity_embedding_stream(
    adata_transvelo,
    basis="umap",
    color="condition",
    ax=axes[2],
    title="TransVelo",
    title_fontsize=18,
    show=False
)

plt.tight_layout()

plt.savefig(
    "3-1/1-velocity_condition.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import scvelo as scv
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1,2, figsize=(12,5))

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="condition",
    groups=["control"],
    ax=axes[0],
    title="Control",
    show=False
)

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="condition",
    groups=["compaction"],
    ax=axes[1],
    title="Compaction",
    show=False
)

plt.tight_layout()

plt.savefig(
    "3-1/condition_velocity_comparison.png",
    dpi=600
)

plt.show()

In [ ]:
from scipy.stats import spearmanr
import numpy as np

def latent_velocity_corr(adata):

    v = np.linalg.norm(
        adata.layers["velocity_model"],
        axis=1
    )

    lt = adata.obs["latent_time"]

    corr, _ = spearmanr(v, lt)

    return corr


corr_scvelo = latent_velocity_corr(adata_scvelo)
corr_deepvelo = latent_velocity_corr(adata_deepvelo)
corr_transvelo = latent_velocity_corr(adata_transvelo)

print(corr_scvelo, corr_deepvelo, corr_transvelo)

In [ ]:
from scipy.stats import spearmanr
import numpy as np

def latent_velocity_corr(adata):

    v = np.linalg.norm(
        adata.layers["velocity_model"],
        axis=1
    )

    lt = adata.obs["latent_time"]

    corr, _ = spearmanr(v, lt)

    return corr


corr_scvelo = latent_velocity_corr(adata_scvelo)
corr_deepvelo = latent_velocity_corr(adata_deepvelo)
corr_transvelo = latent_velocity_corr(adata_transvelo)

print(corr_scvelo, corr_deepvelo, corr_transvelo)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.DataFrame({
    "model":["scVelo","DeepVelo","TransVelo"],
    "correlation":[
        corr_scvelo,
        corr_deepvelo,
        corr_transvelo
    ]
})

plt.figure(figsize=(6,5))

sns.barplot(data=df, x="model", y="correlation")

plt.ylabel("Spearman correlation")

plt.title("Velocity–Latent Time Correlation")

plt.tight_layout()

plt.savefig(
    "Results/latent_time_correlation.png",
    dpi=600
)

plt.show()

In [ ]:
#11 Velocity Strength
df = pd.concat([

    pd.DataFrame({"model":"scVelo","value":adata_scvelo.obs["velocity_strength"]}),
    pd.DataFrame({"model":"DeepVelo","value":adata_deepvelo.obs["velocity_strength"]}),
    pd.DataFrame({"model":"TransVelo","value":adata_transvelo.obs["velocity_strength"]})

])

sns.boxplot(data=df, x="model", y="value")

plt.title("Velocity Strength")

plt.savefig("Results/2_velocity_strength.png", dpi=600)

plt.show()

In [ ]:
#12 Velocity Confidence
df = pd.concat([

    pd.DataFrame({"model":"scVelo","value":adata_scvelo.obs["velocity_confidence"]}),
    pd.DataFrame({"model":"DeepVelo","value":adata_deepvelo.obs["velocity_confidence"]}),
    pd.DataFrame({"model":"TransVelo","value":adata_transvelo.obs["velocity_confidence"]})

])

sns.boxplot(data=df, x="model", y="value")

plt.title("Velocity Confidence")

plt.savefig("Results/3_velocity_confidence.png", dpi=600)

plt.show()

In [ ]:
#13 Velocity Coherence
df = pd.concat([

    pd.DataFrame({"model":"scVelo","value":adata_scvelo.obs["velocity_coherence"]}),
    pd.DataFrame({"model":"DeepVelo","value":adata_deepvelo.obs["velocity_coherence"]}),
    pd.DataFrame({"model":"TransVelo","value":adata_transvelo.obs["velocity_coherence"]})

])

sns.boxplot(data=df, x="model", y="value")

plt.title("Velocity Coherence")

plt.savefig("Results/5_velocity_coherence.png", dpi=600)

plt.show()

In [ ]:
#14 Trajectory Smoothness
df = pd.concat([

    pd.DataFrame({"model":"scVelo","value":adata_scvelo.obs["trajectory_smoothness"]}),
    pd.DataFrame({"model":"DeepVelo","value":adata_deepvelo.obs["trajectory_smoothness"]}),
    pd.DataFrame({"model":"TransVelo","value":adata_transvelo.obs["trajectory_smoothness"]})

])

sns.boxplot(data=df, x="model", y="value")

plt.title("Trajectory Smoothness")

plt.savefig("Results/6_trajectory_smoothness.png", dpi=600)

plt.show()

In [ ]:
#15 Velocity-Pseudotime Correlation
df = pd.DataFrame({

    "model":["scVelo","DeepVelo","TransVelo"],

    "correlation":[
        velocity_pt_corr(adata_scvelo),
        velocity_pt_corr(adata_deepvelo),
        velocity_pt_corr(adata_transvelo)
    ]
})

sns.barplot(data=df, x="model", y="correlation")

plt.title("Velocity-Pseudotime Correlation")

plt.savefig("Results/7_velocity_pseudotime_corr.png", dpi=600)

plt.show()

In [ ]:
#16 Benchmark Table
benchmark = pd.DataFrame({

    "model":[
        "scVelo",
        "DeepVelo",
        "TransVelo"
    ],

    "velocity_strength":[
        adata_scvelo.obs["velocity_strength"].mean(),
        adata_deepvelo.obs["velocity_strength"].mean(),
        adata_transvelo.obs["velocity_strength"].mean()
    ],

    "velocity_confidence":[
        adata_scvelo.obs["velocity_confidence"].mean(),
        adata_deepvelo.obs["velocity_confidence"].mean(),
        adata_transvelo.obs["velocity_confidence"].mean()
    ],

    "velocity_coherence":[
        adata_scvelo.obs["velocity_coherence"].mean(),
        adata_deepvelo.obs["velocity_coherence"].mean(),
        adata_transvelo.obs["velocity_coherence"].mean()
    ],

    "trajectory_smoothness":[
        adata_scvelo.obs["trajectory_smoothness"].mean(),
        adata_deepvelo.obs["trajectory_smoothness"].mean(),
        adata_transvelo.obs["trajectory_smoothness"].mean()
    ],

    "pseudotime_variance":[
        adata_scvelo.obs["velocity_pseudotime"].var(),
        adata_deepvelo.obs["velocity_pseudotime"].var(),
        adata_transvelo.obs["velocity_pseudotime"].var()
    ],

    "velocity_pseudotime_corr":[
        velocity_pt_corr(adata_scvelo),
        velocity_pt_corr(adata_deepvelo),
        velocity_pt_corr(adata_transvelo)
    ]

})

benchmark.to_csv("Results/benchmark_table.csv", index=False)

print(benchmark)

In [ ]:
#17 新加的指标，CBDC（Cross-Boundary Direction Correctness）
def cross_boundary_correctness(
    adata,
    cluster_key,
    transitions
):

    V = adata.layers["velocity_model"]

    neighbors = adata.obsp["connectivities"]

    clusters = adata.obs[cluster_key]

    scores = []

    for source, target in transitions:

        idx_source = np.where(clusters == source)[0]

        idx_target = np.where(clusters == target)[0]

        target_cells = adata.obsm["X_umap"][idx_target]

        source_cells = adata.obsm["X_umap"][idx_source]

        V_source = V[idx_source]

        score = []

        for i, v in enumerate(V_source):

            direction = target_cells.mean(axis=0) - source_cells[i]

            cos_sim = np.dot(v[:2], direction) / (
                np.linalg.norm(v[:2]) *
                np.linalg.norm(direction) + 1e-8
            )

            score.append(cos_sim)

        scores.append(np.nanmean(score))

    return np.mean(scores)


In [ ]:
transitions = [

    ("stem","cortex"),

    ("stem","epidermis"),

    ("stem","endodermis")

]

cluster_key="celltype.anno"

In [ ]:
cbdc_scvelo = cross_boundary_correctness(
    adata_scvelo,
    "celltype.anno",
    transitions
)

cbdc_deepvelo = cross_boundary_correctness(
    adata_deepvelo,
    "celltype.anno",
    transitions
)

cbdc_transvelo = cross_boundary_correctness(
    adata_transvelo,
    "celltype.anno",
    transitions
)

In [ ]:
#18 Velocity Consistency Score
def velocity_consistency(adata):

    V = adata.layers["velocity_model"]

    neighbors = adata.obsp["connectivities"]

    consistency = []

    for i in range(V.shape[0]):

        neigh_idx = neighbors[i].indices

        if len(neigh_idx) == 0:

            consistency.append(np.nan)

            continue

        v = V[i]

        v_neighbors = V[neigh_idx]

        diff = np.linalg.norm(v_neighbors - v, axis=1)

        consistency.append(np.mean(diff))

    return np.nanmean(consistency)

In [ ]:
cons_scvelo = velocity_consistency(adata_scvelo)

cons_deepvelo = velocity_consistency(adata_deepvelo)

cons_transvelo = velocity_consistency(adata_transvelo)

In [ ]:
benchmark = pd.DataFrame({

    "model":[
        "scVelo",
        "DeepVelo",
        "TransVelo"
    ],

    "velocity_strength":[
        adata_scvelo.obs["velocity_strength"].mean(),
        adata_deepvelo.obs["velocity_strength"].mean(),
        adata_transvelo.obs["velocity_strength"].mean()
    ],

    "velocity_confidence":[
        adata_scvelo.obs["velocity_confidence"].mean(),
        adata_deepvelo.obs["velocity_confidence"].mean(),
        adata_transvelo.obs["velocity_confidence"].mean()
    ],

    "velocity_coherence":[
        adata_scvelo.obs["velocity_coherence"].mean(),
        adata_deepvelo.obs["velocity_coherence"].mean(),
        adata_transvelo.obs["velocity_coherence"].mean()
    ],

    "trajectory_smoothness":[
        adata_scvelo.obs["trajectory_smoothness"].mean(),
        adata_deepvelo.obs["trajectory_smoothness"].mean(),
        adata_transvelo.obs["trajectory_smoothness"].mean()
    ],

    "pseudotime_variance":[
        adata_scvelo.obs["velocity_pseudotime"].var(),
        adata_deepvelo.obs["velocity_pseudotime"].var(),
        adata_transvelo.obs["velocity_pseudotime"].var()
    ],

    "velocity_pseudotime_corr":[
        velocity_pt_corr(adata_scvelo),
        velocity_pt_corr(adata_deepvelo),
        velocity_pt_corr(adata_transvelo)
    ],

    "CBDC":[
        cbdc_scvelo,
        cbdc_deepvelo,
        cbdc_transvelo
    ],

    "velocity_consistency":[
        cons_scvelo,
        cons_deepvelo,
        cons_transvelo
    ]

})

print(benchmark)

benchmark.to_csv(
    "Results/benchmark_table_full.csv",
    index=False
)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

def velocity_driver_genes(adata):

    V = adata.layers["velocity_model"]
    X = adata.layers["Ms"] if "Ms" in adata.layers else adata.X

    scores = []

    for g in range(V.shape[1]):

        v = V[:, g]
        x = X[:, g]

        if np.std(v) == 0 or np.std(x) == 0:
            scores.append(0)
            continue

        corr, _ = spearmanr(v, x)

        scores.append(abs(corr))

    scores = np.array(scores)

    adata.var["velocity_driver_score"] = scores

    return scores

In [ ]:
velocity_driver_genes(adata_transvelo)

In [ ]:
top_genes = (
    adata_transvelo.var
    .sort_values("velocity_driver_score", ascending=False)
    .head(20)
    .index
)

In [ ]:
adata_sorted = adata_transvelo[
    adata_transvelo.obs["velocity_pseudotime"].argsort()
]

In [ ]:
expr = adata_sorted[:, top_genes].X

if not isinstance(expr, np.ndarray):
    expr = expr.toarray()

expr = (expr - expr.mean(axis=0)) / expr.std(axis=0)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

sns.heatmap(
    expr.T,
    cmap="viridis",
    yticklabels=top_genes,
    xticklabels=False
)

plt.ylabel("Driver Genes")
plt.xlabel("Cells ordered by velocity pseudotime")

plt.title("TransVelo Driver Genes")

plt.tight_layout()

plt.savefig(
    "Results/10_driver_gene_heatmap.png",
    dpi=600
)

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

for g in top_genes[:5]:

    scv.pl.scatter(
        adata_transvelo,
        x="velocity_pseudotime",
        y=g,
        color="celltype.anno",
        show=False
    )

plt.savefig(
    "Results/11_driver_gene_trends.png",
    dpi=600
)